In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix

from sklearn.preprocessing import binarize

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'

In [ ]:
mtx = mmread(f'{projdir}/mtx/all_raw.mtx')

In [ ]:
mtx

In [ ]:
genes = pd.read_csv(f'{projdir}/csv/rna/all_rna_genes.csv', sep='\t', header=None, index_col=0)
nuclei = pd.read_csv(f'{projdir}/csv/rna/all_rna_nuclei.csv', sep='\t', header=None, index_col=0)

In [ ]:
mtx = csr_matrix(mtx)

In [ ]:
adata = anndata.AnnData(X=mtx, obs=genes, var=nuclei).T

In [ ]:
adata

In [ ]:
adata.obs['donor'] = [x.split('-')[1][:3] for x in adata.obs.index]
adata.obs['time'] = [x.split('-')[1][3:] for x in adata.obs.index]
adata.obs['time_num'] = [int(x[1:]) for x in adata.obs['time']]

In [ ]:
adata

In [ ]:
### metadata

meta_path = '/u/project/cluo_scratch/chliu/IGVF_iPSC_PGP_Transcriptome/Output/'
meta_in = f'{meta_path}/20240425_rna_metadata.tsv'

rna_meta = pd.read_csv(meta_in, sep='\t', header=0, index_col=0)
rna_meta.shape

In [ ]:
rna_meta.head()

In [ ]:
adata.obs['traj_score'] = rna_meta.reindex(adata.obs.index)['Trajectory Score']

In [ ]:
adata.shape

In [ ]:
mask = adata.obs['traj_score'].isna()
adata = adata[~mask]

In [ ]:
adata

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
multiome_raw = sc.read_h5ad(f'{indir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
multiome_raw.shape

### plot some basic statistics
- amount of overlapping genes
- counts distribution per gene

In [ ]:
multiome_raw.var

In [ ]:
adata.var.head()

In [ ]:
indir = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations'
gene_annot = pd.read_csv(f'{indir}/ensembl_to_symbol.tsv', sep='\t', header=0, index_col=None)
gene_to_symbol = dict(zip(gene_annot['gene'], gene_annot['symbol']))
symbol_to_gene = dict(zip(gene_annot['symbol'], gene_annot['gene']))


In [ ]:
adata.var['symbol'] = [gene_to_symbol.get(x) for x in adata.var.index]
adata.var.head()

In [ ]:
multiome_raw.var['ensg'] = [symbol_to_gene.get(x) for x in multiome_raw.var.index]

### reindex to have the same gene order

In [ ]:
mct_mask = adata.var.index.isin(multiome_raw.var['ensg'])
mult_mask = multiome_raw.var['ensg'].isin(adata.var.index)
sum(mct_mask), sum(mult_mask)

In [ ]:
sub_mct = adata[:, mct_mask]
sub_mult = multiome_raw[:, mult_mask]

In [ ]:
sub_mct.var.head()

In [ ]:
sub_mult.var.head()

In [ ]:
mct_mat = pd.DataFrame(adata.X.toarray(), columns=adata.var.index, index=adata.obs.index)

In [ ]:
mct_mat.head()

In [ ]:
t_mct = mct_mat.T
t_mct.head()

In [ ]:
mct_reindexed = t_mct.reindex(sub_mult.var['ensg'])
mct_reindexed.head()

In [ ]:
var_reindexed = adata.var.reindex(sub_mult.var['ensg'])
var_reindexed.head()

In [ ]:
final_multiome = sub_mult.copy()

In [ ]:
final_mat = csr_matrix(mct_reindexed.values.T)

In [ ]:
final_mct = anndata.AnnData(X=final_mat, obs=adata.obs, var = var_reindexed)
final_mct.var['ensg'] = final_mct.var.index
final_mct.var.index = final_mct.var['symbol']

In [ ]:
final_mct

In [ ]:
final_mct.shape, final_multiome.shape

In [ ]:
final_mct.var.head()

In [ ]:
final_multiome.var.head()

In [ ]:
sum(final_mct.var.index == final_multiome.var.index)

### make sure it's raw counts

In [ ]:
final_mct.X[:100, :100].toarray()

In [ ]:
final_multiome.X[:100,:100].toarray()

In [ ]:
final_mct.obs.index.name = 'nucleus'

In [ ]:
final_mct.write_h5ad(f'{projdir}/h5ad/raw/mct.filter.h5ad')
final_multiome.write_h5ad(f'{projdir}/h5ad/raw/multiome.filter.h5ad')

In [ ]:
!date